# Deploy Azure Front Door with Custom Domain, WAF Bot Protection & Hello World Backend

This notebook deploys the following Azure infrastructure using **Bicep AVM (Azure Verified Modules)**:

| Resource | AVM Module | Purpose |
|----------|-----------|---------|
| **Azure Front Door Premium** | `avm/res/cdn/profile` | CDN/Front Door profile with endpoint, origin group, routes |
| **WAF Policy** | `avm/res/network/front-door-web-application-firewall-policy` | Bot protection using `Microsoft_BotManagerRuleSet` |
| **Container App** | `avm/res/app/container-app` | Hello World backend origin |
| **Container App Environment** | `avm/res/app/managed-environment` | Hosting environment for Container Apps |

### Architecture
```
User → Azure Front Door (WAF Bot Protection) → Custom Domain → Origin Group → Container App (Hello World)
```

### Prerequisites
- Azure CLI installed
- Bicep CLI installed
- An Azure subscription with Contributor access
- A custom domain you own (for DNS validation)

## 1. Install and Verify Prerequisites

Check that Azure CLI and Bicep CLI are installed and display their versions.

In [ ]:
import subprocess
import json
import sys

def run_cmd(cmd, check=True):
    """Run a shell command and return the result."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}")
    return result

# Check Azure CLI version
print("=== Azure CLI Version ===")
run_cmd("az version --output table")

# Check Bicep CLI version
print("\n=== Bicep CLI Version ===")
run_cmd("az bicep version")

In [ ]:
# Authenticate to Azure (opens browser for interactive login)
run_cmd("az login --output table")

# Show current subscription
print("\n=== Current Subscription ===")
run_cmd("az account show --output table")

## 2. Define Deployment Configuration

Set the deployment parameters. **Update `CUSTOM_DOMAIN` with your actual domain name.**

In [ ]:
# ============================================================
# CONFIGURATION - Update these values for your environment
# ============================================================

RESOURCE_GROUP = "rg-frontdoor-helloworld"
LOCATION = "eastus2"
CUSTOM_DOMAIN = "www.contoso.com"       # <-- CHANGE THIS to your domain
DEPLOYMENT_NAME = "afd-helloworld-deploy"
BICEP_TEMPLATE = "infra/main.bicep"
NAME_PREFIX = "afd"

print(f"Resource Group:  {RESOURCE_GROUP}")
print(f"Location:        {LOCATION}")
print(f"Custom Domain:   {CUSTOM_DOMAIN}")
print(f"Bicep Template:  {BICEP_TEMPLATE}")
print(f"Name Prefix:     {NAME_PREFIX}")

## 3. Review the Bicep Template (AVM Modules)

The Bicep template at `infra/main.bicep` uses the following **Azure Verified Modules (AVM)**:

| AVM Module | Registry Reference | Purpose |
|------------|-------------------|---------|
| WAF Policy | `br/public:avm/res/network/front-door-web-application-firewall-policy:0.3.1` | WAF with `Microsoft_BotManagerRuleSet` v1.1 + `Microsoft_DefaultRuleSet` v2.1 |
| Front Door Profile | `br/public:avm/res/cdn/profile:0.12.0` | Front Door Premium with endpoints, origin groups, custom domains, security policies |
| Container App Environment | `br/public:avm/res/app/managed-environment:0.8.1` | Managed environment for Container Apps |
| Container App | `br/public:avm/res/app/container-app:0.12.0` | Hello World container as the backend origin |

### WAF Bot Protection Configuration
The WAF policy includes:
- **`Microsoft_BotManagerRuleSet` v1.1** - Detects and blocks malicious bots
- **`Microsoft_DefaultRuleSet` v2.1** - OWASP-based core rule set for additional protection
- **Prevention mode** - Blocks matching requests (not just detection)

In [ ]:
# Display the Bicep template for review
with open(BICEP_TEMPLATE, "r") as f:
    print(f.read())

## 4. Create Resource Group

In [ ]:
# Create the resource group
run_cmd(f"az group create --name {RESOURCE_GROUP} --location {LOCATION} --output table")

## 5. Restore AVM Bicep Modules from Registry

Download the AVM modules from the Bicep public module registry.

In [ ]:
# Restore (download) AVM modules from the Bicep public registry
run_cmd(f"az bicep restore --file {BICEP_TEMPLATE} --force")

## 6. Compile and Validate the Bicep Template

Build the Bicep to ARM JSON and run preflight validation against Azure.

In [ ]:
# Build - compile Bicep to ARM JSON to check for syntax errors
print("=== Compiling Bicep Template ===")
run_cmd(f"az bicep build --file {BICEP_TEMPLATE}")
print("Bicep compilation successful!\n")

# Validate - preflight validation against Azure
print("=== Validating Deployment ===")
run_cmd(
    f"az deployment group validate "
    f"--resource-group {RESOURCE_GROUP} "
    f"--template-file {BICEP_TEMPLATE} "
    f"--parameters customDomainHostName={CUSTOM_DOMAIN} namePrefix={NAME_PREFIX} "
    f"--output table"
)
print("Validation passed!")

## 7. Deploy the Bicep Template to Azure

This deploys all resources:
1. **Log Analytics Workspace** - Required by Container Apps
2. **Container App Environment** + **Hello World Container App** - Backend origin
3. **WAF Policy** - Bot Manager + Default Rule Sets in Prevention mode
4. **Front Door Premium Profile** - Endpoint, origin group, custom domain, routes, security policy

> **Note:** Deployment takes approximately 5-10 minutes.

In [ ]:
# Deploy the infrastructure
print("=== Starting Deployment ===\n")
result = run_cmd(
    f"az deployment group create "
    f"--resource-group {RESOURCE_GROUP} "
    f"--name {DEPLOYMENT_NAME} "
    f"--template-file {BICEP_TEMPLATE} "
    f"--parameters customDomainHostName={CUSTOM_DOMAIN} namePrefix={NAME_PREFIX} "
    f"--output json"
)

# Parse and display outputs
if result.stdout:
    deployment = json.loads(result.stdout)
    outputs = deployment.get("properties", {}).get("outputs", {})
    print("\n=== Deployment Outputs ===")
    for key, value in outputs.items():
        print(f"  {key}: {value.get('value', 'N/A')}")

## 8. DNS Configuration for Custom Domain

After deployment, you must configure DNS records for your custom domain.

| Record Type | Name | Value |
|-------------|------|-------|
| **CNAME** | `www` (or your subdomain) | `<endpoint-hostname>.z01.azurefd.net` |
| **TXT** | `_dnsauth.<your-domain>` | *(validation token from Azure)* |

Run the cell below to get the exact values for your deployment.

In [ ]:
# Get Front Door profile and endpoint details for DNS configuration
import hashlib

unique_suffix = hashlib.md5(RESOURCE_GROUP.encode()).hexdigest()[:13]  # approximate uniqueString

# List Front Door profiles in the resource group
print("=== Front Door Profiles ===")
result = run_cmd(
    f"az afd profile list "
    f"--resource-group {RESOURCE_GROUP} "
    f"--output table"
)

# List Front Door endpoints
print("\n=== Front Door Endpoints ===")
profiles_result = run_cmd(
    f"az afd profile list "
    f"--resource-group {RESOURCE_GROUP} "
    f"--output json",
    check=False
)
if profiles_result.stdout:
    profiles = json.loads(profiles_result.stdout)
    if profiles:
        profile_name = profiles[0]["name"]
        run_cmd(
            f"az afd endpoint list "
            f"--resource-group {RESOURCE_GROUP} "
            f"--profile-name {profile_name} "
            f"--output table"
        )

        # Get custom domain validation info
        print("\n=== Custom Domain Validation Info ===")
        result = run_cmd(
            f"az afd custom-domain list "
            f"--resource-group {RESOURCE_GROUP} "
            f"--profile-name {profile_name} "
            f"--output json",
            check=False
        )
        if result.stdout:
            domains = json.loads(result.stdout)
            for domain in domains:
                print(f"\nDomain: {domain.get('hostName', 'N/A')}")
                validation = domain.get("validationProperties", {})
                if validation:
                    print(f"  TXT Record Name:  _dnsauth.{domain.get('hostName', '')}")
                    print(f"  TXT Record Value: {validation.get('validationToken', 'N/A')}")
                    print(f"  Expiry:           {validation.get('expirationDate', 'N/A')}")
                print(f"  Domain Status:    {domain.get('domainValidationState', 'N/A')}")

## 9. Verify Deployment and Test Endpoint

Verify all deployed resources and test the Front Door endpoint.

In [ ]:
# Verify all deployed resources
print("=== Deployed Resources ===")
run_cmd(
    f"az resource list "
    f"--resource-group {RESOURCE_GROUP} "
    f"--output table "
    f"--query \"[].{{Name:name, Type:type, Location:location}}\""
)

# Verify WAF Policy
print("\n=== WAF Policy Details ===")
waf_result = run_cmd(
    f"az network front-door waf-policy list "
    f"--resource-group {RESOURCE_GROUP} "
    f"--output table",
    check=False
)

# Verify Container App
print("\n=== Container App ===")
run_cmd(
    f"az containerapp list "
    f"--resource-group {RESOURCE_GROUP} "
    f"--output table",
    check=False
)

In [ ]:
import urllib.request
import urllib.error

# Get the Front Door endpoint hostname
profiles_result = run_cmd(
    f"az afd profile list "
    f"--resource-group {RESOURCE_GROUP} "
    f"--output json",
    check=False
)

if profiles_result.stdout:
    profiles = json.loads(profiles_result.stdout)
    if profiles:
        profile_name = profiles[0]["name"]
        ep_result = run_cmd(
            f"az afd endpoint list "
            f"--resource-group {RESOURCE_GROUP} "
            f"--profile-name {profile_name} "
            f"--output json",
            check=False
        )
        if ep_result.stdout:
            endpoints = json.loads(ep_result.stdout)
            if endpoints:
                endpoint_hostname = endpoints[0]["hostName"]
                endpoint_url = f"https://{endpoint_hostname}"

                print(f"=== Testing Front Door Endpoint ===")
                print(f"URL: {endpoint_url}\n")

                # Test with normal user agent
                try:
                    req = urllib.request.Request(endpoint_url)
                    req.add_header("User-Agent", "Mozilla/5.0")
                    with urllib.request.urlopen(req, timeout=30) as response:
                        print(f"Status: {response.status}")
                        print(f"Headers: {dict(response.headers)}")
                        body = response.read().decode("utf-8")
                        print(f"Body (first 500 chars): {body[:500]}")
                except urllib.error.HTTPError as e:
                    print(f"HTTP Error: {e.code} - {e.reason}")
                except Exception as e:
                    print(f"Connection error (endpoint may still be provisioning): {e}")

                # Test with bot-like user agent to verify WAF
                print("\n=== Testing WAF Bot Detection ===")
                try:
                    req = urllib.request.Request(endpoint_url)
                    req.add_header("User-Agent", "python-requests/2.28.0")
                    with urllib.request.urlopen(req, timeout=30) as response:
                        print(f"Status: {response.status}")
                        print("Note: Bot traffic may be blocked or redirected by WAF")
                except urllib.error.HTTPError as e:
                    print(f"HTTP Error: {e.code} - Bot traffic blocked by WAF (expected behavior)")
                except Exception as e:
                    print(f"Connection error: {e}")

## 10. Cleanup

Run the cell below to delete all deployed resources. **This action is irreversible.**

In [ ]:
# DELETE all resources - uncomment and run when done
# run_cmd(f"az group delete --name {RESOURCE_GROUP} --yes --no-wait")
# print(f"Resource group '{RESOURCE_GROUP}' deletion initiated.")